In [1]:
import os, shutil

def find(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

src = find("11_train_student_lat.py")
shutil.copy(src, "/kaggle/working/11_train_student_lat.py")
print("Script:", src)
for f in ["_meta_crop.csv", "_meta_280.csv", "knee_side.csv", "oof_effb0_kf2.csv", "report_labels.csv"]:
    print(f"{f:20s} -> {find(f)}")

Script: /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/11_train_student_lat.py
_meta_crop.csv       -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-27-preprocess-crop/vol_train_crop/_meta_crop.csv
_meta_280.csv        -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-28-preprocess-280/vol_train_280/_meta_280.csv
knee_side.csv        -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-22-sisi-lutut/knee_side.csv
oof_effb0_kf2.csv    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-06-tahapc-teacher-oof/oof_effb0_kf2.csv
report_labels.csv    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --oof oof_effb0_kf2.csv --side_csv knee_side.csv \
        --backbone vit_small_patch14_dinov2.lvd142m --lr_backbone 1e-4 --llrd 0.8 --epochs 12 --seed 42"

# GPU 0: crop 288 -> 224, masukan DINOv2 tetap 224
CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $COMMON \
    --meta _meta_crop.csv --batch 4 --tag dinov2s_student_v1d_lat_crop \
    > log_dinov2s_student_v1d_lat_crop.txt 2>&1 &

# GPU 1: gambar utuh 280 px, masukan DINOv2 280 (batch 2 x akumulasi 2 = setara batch 4, hemat memori)
CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $COMMON \
    --meta _meta_280.csv --img_size 280 --batch 2 --accum 2 --tag dinov2s_student_v1d_lat_280 \
    > log_dinov2s_student_v1d_lat_280.txt 2>&1 &
wait
for t in dinov2s_student_v1d_lat_crop dinov2s_student_v1d_lat_280; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Volume\|Normalisasi\|Train \|ukuran\|Error\|error" log_*.txt

===== dinov2s_student_v1d_lat_crop =====
Epoch 11: loss 0.3726 | val AUC 0.8838 | val-ref AUC 0.8979 | gold AUC 0.8670 | 14.2 menit

=== [dinov2s_student_v1d_lat_crop] Checkpoint terbaik (val-ref AUC 0.8979) pada label emas ===
ACL                AUC=0.912
MCL                AUC=0.923
Medial Meniscus    AUC=0.865
Lateral Meniscus   AUC=0.804
Medial OA          AUC=0.967
Lateral OA         AUC=0.827
PF OA              AUC=0.811
Effusion           AUC=0.857
Synovitis          AUC=0.726
Baker's            AUC=0.949
Contusion          AUC=0.884
Fracture           AUC=0.878
Mean AUC gold: 0.8670
===== dinov2s_student_v1d_lat_280 =====
Epoch 11: loss 0.3718 | val AUC 0.8848 | val-ref AUC 0.8984 | gold AUC 0.8681 | 22.3 menit

=== [dinov2s_student_v1d_lat_280] Checkpoint terbaik (val-ref AUC 0.8985) pada label emas ===
ACL                AUC=0.915
MCL                AUC=0.912
Medial Meniscus    AUC=0.851
Lateral Meniscus   AUC=0.750
Medial OA          AUC=0.981
Lateral OA         AUC=0.814
PF

In [3]:
import pandas as pd
for tag in ["dinov2s_student_v1d_lat_crop", "dinov2s_student_v1d_lat_280"]:
    print(f"\n===== {tag} =====")
    try:
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== dinov2s_student_v1d_lat_crop =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5372   0.7791       0.7949    0.8099 14.1270
     1 0.4628   0.8256       0.8420    0.8533 14.2161
     2 0.4414   0.8508       0.8656    0.8389 14.1471
     3 0.4307   0.8604       0.8746    0.8536 14.1322
     4 0.4212   0.8605       0.8749    0.8468 14.1420
     5 0.4125   0.8773       0.8910    0.8588 14.1137
     6 0.4032   0.8773       0.8920    0.8770 14.1999
     7 0.3943   0.8809       0.8956    0.8687 14.1652
     8 0.3861   0.8824       0.8961    0.8671 14.1329
     9 0.3791   0.8823       0.8971    0.8658 14.2917
    10 0.3749   0.8838       0.8978    0.8660 14.2885
    11 0.3726   0.8838       0.8979    0.8670 14.2414

===== dinov2s_student_v1d_lat_280 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5384   0.7644       0.7764    0.7969 22.3527
     1 0.4671   0.8270       0.8410    0.8396 22.3081
     2 0.4441   0.8383       0.8556    0.8472 22.2808